In [1]:
import numpy as np
import time

import sys

module_dir = r"src"
sys.path.append(module_dir)
import PhotonGenerationFunctions

PGF = PhotonGenerationFunctions.PhotonGenFunctions()
import matplotlib.pyplot as plt
import IOFunctions

IO = IOFunctions.IO_Functions()
import PlottingFunctions

plots = PlottingFunctions.Plotter()

import os

from src import Multicolour_Simulation_Functions

MSF = Multicolour_Simulation_Functions.MultiC_Sim_Funcs()
import polars as pl

In [7]:
# input camera params
data_folder = "/home/jbeckwith/Documents/Dropbox/Cambridge University Dropbox/Joseph Beckwith/Chemistry/Lee/Data/Salix/CS505CU_Calibration"
gain = IO.read_tiff(os.path.join(data_folder, "gain.tif"))
offset = IO.read_tiff(os.path.join(data_folder, "offset.tif"))
variance = IO.read_tiff(os.path.join(data_folder, "variance.tif"))
readnoise = IO.read_tiff(os.path.join(data_folder, "readnoise.tif"))
rqe = IO.read_tiff(os.path.join(data_folder, "rqe.tif"))

image_size = 40
from Camera_QE import getpixelefficiency

gpe = getpixelefficiency.GPE()
R, G, B, wavelength = gpe.getpixelefficiency("Camera_QE/CS505CU_QE.csv")
masks = MSF.MF.get_masks(MSF.mosaic_unit, image_size, image_size)

absolute_QYs = np.vstack([B, G, R])
camera_calibration = {}
camera_calibration["gain"] = gain[:image_size, :image_size]
camera_calibration["offset"] = offset[:image_size, :image_size]
camera_calibration["variance"] = variance[:image_size, :image_size]
camera_calibration["readnoise"] = readnoise[:image_size, :image_size]
camera_calibration["rqe"] = rqe[:image_size, :image_size]

In [3]:
dye_names = np.array(
    [
        "ATTO488",
        "AF514",
        "Cy3B",
        "ATTO565",
        "ATTO590",
        "ATTO610",
        "ATTO647N",
        "AF700",
        "ATTO740",
    ]
)

In [4]:
folder = "Spectra/Em/Dyes/"
emission_spectra = np.sort(os.listdir(folder))
dyes = np.zeros([len(dye_names), len(wavelength)])

for i, dye in enumerate(dye_names):
    file = np.sort([e for e in emission_spectra if dye in e])[0]
    skip_rows = 1
    if "ATTO" in dye:
        separator = "\t"
    else:
        separator = ","

    data = pl.read_csv(
        os.path.join(folder, file), separator=separator, skip_rows=skip_rows
    )
    d_wl = data[:, 0].to_numpy()
    d_em = data[:, 1].to_numpy()
    dye_spectrum = np.interp(x=wavelength, xp=d_wl, fp=d_em)
    dye_spectrum = dye_spectrum / np.sum(dye_spectrum)
    dyes[i, :] = dye_spectrum

In [5]:
kon = 37e6
conc = 5000e-12
frametime = 100e-3
photonrate = 4000
photonbudget = 1500000
photonratestd = 8000
taud = 1 / (kon * conc)  # in seconds
bright_time = 200e-3
frames = 200000

In [8]:
linker_length_array = np.array([20, 40, 80, 300, 600])
start = time.time()
for i, linker_length in enumerate(linker_length_array):
    save_folder = "/home/jbeckwith/Documents/Dropbox/Cambridge University Dropbox/Joseph Beckwith/Chemistry/Lee/Data/Simulation/20241025/data"
    save_name = (
        "GIF_Test_9colourPAINT_linkerlength_pixelsize69nm_"
        + str(int(linker_length))
        + "nm"
    )
    save_overall_name = os.path.join(save_folder, save_name)

    image_stack, spot_locations = PGF.PAINT_array_generator(
        save_overall_name,
        camera_calibration,
        wavelength,
        absolute_QYs,
        dyes,
        dye_names,
        kon=kon,
        conc=conc,
        bright_time=300e-3,
        frames=1000,
        frametime=frametime,
        linker_size=linker_length,
        photonrate=4000,
        photonratestd=8000,
        photonbudget=1500000,
        pixel_size=69,
        grid_n=3,
        NA=1.49,
        background_photons=0,
        gen_trace_function=True,
    )
    print(
        "Simulated PAINT stack {}/{}    Time elapsed: {:.3f} min".format(
            i + 1, len(linker_length_array), (time.time() - start) / 60.0
        ),
        end="\r",
        flush=True,
    )
    del image_stack
    del spot_locations

In [7]:
linker_length_array = np.array([20, 40, 80, 300, 600])
start = time.time()
for i, linker_length in enumerate(linker_length_array):
    save_folder = "/home/jbeckwith/Documents/Dropbox/Cambridge University Dropbox/Joseph Beckwith/Chemistry/Lee/Data/Simulation/20241025/data"
    save_name = (
        "Test_9colourPAINT_linkerlength_pixelsize69nm_" + str(int(linker_length)) + "nm"
    )
    save_overall_name = os.path.join(save_folder, save_name)

    image_stack, spot_locations = PGF.PAINT_array_generator(
        save_overall_name,
        camera_calibration,
        wavelength,
        absolute_QYs,
        dyes,
        dye_names,
        kon=kon,
        conc=conc,
        bright_time=300e-3,
        frames=frames,
        frametime=frametime,
        linker_size=linker_length,
        photonrate=4000,
        photonratestd=8000,
        photonbudget=1500000,
        pixel_size=69,
        grid_n=3,
        NA=1.49,
        background_photons=0,
        gen_trace_function=False,
    )
    print(
        "Simulated PAINT stack {}/{}    Time elapsed: {:.3f} min".format(
            i + 1, len(linker_length_array), (time.time() - start) / 60.0
        ),
        end="\r",
        flush=True,
    )
    del image_stack
    del spot_locations

In [ ]:
linker_length = 600
grid_n = 3
pixel_size = 140
w = 40
h = 40

In [ ]:
x_centres = np.add(
    (
        np.arange(0, linker_length * grid_n, linker_length)
        - np.mean(np.arange(0, linker_length * grid_n, linker_length))
    ),
    np.divide(np.multiply(pixel_size, w), 2),
)
y_centres = np.add(
    (
        np.arange(0, linker_length * grid_n, linker_length)
        - np.mean(np.arange(0, linker_length * grid_n, linker_length))
    ),
    np.divide(np.multiply(pixel_size, h), 2),
)

In [ ]:
x0, y0 = np.meshgrid(x_centres, y_centres)

In [ ]:
from src import PlottingFunctions

plotter = PlottingFunctions.Plotter()

In [ ]:
fig, axs = plotter.one_column_plot()

cs = [
    "#c06a5a",
    "#d14c38",
    "#d09a41",
    "#838139",
    "#76b341",
    "#51a876",
    "#4bafd0",
    "#7178ca",
    "#c75a8e",
]

labels = [
    "ATTO 488",
    "AF 514",
    "Cy3B",
    "ATTO 565",
    "ATTO 590",
    "ATTO 610",
    "ATTO 647N",
    "AF 700",
    "ATTO 740",
]

cs = cs[::-1]

axs = plotter.image_plot(
    axs=axs,
    data=np.zeros([6000, 6000]),
    vmin=0,
    vmax=1,
    pixelsize=140 / (6000 / 40),
    scalebarlabel="300 nm",
    scalebarsize=300,
)

for i in np.arange(len(cs)):
    plt.scatter(x0.ravel()[i], y0.ravel()[i], s=5, color=cs[i], label=labels[i])

plt.legend(loc="best", fontsize=6)

# plt.xlim([2600, 3000])
# plt.ylim([2600, 3000])

folder = "/home/jbeckwith/Documents/Dropbox/Cambridge University Dropbox/Joseph Beckwith/Chemistry/Lee/Talks+Posters/Group Meeting/20241104/fig/multicolour_camera/hypothetical_PAINT"
plt.savefig(os.path.join(folder, "600nm_DyeArrangement.svg"), format="svg", dpi=600)

In [ ]:
x0